In [1]:
import trafilatura
import json
from langdetect import detect
import hanzidentifier
from urllib.parse import urlparse
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [2]:
def extract_TUN_XIAO_EDU_AU_link(max_pages=14):
    links = set()
    for page_number in range(max_pages):
        if page_number == 0:
            url = "https://au.oliuxue.com/studentnews/"
        else:
            url = f"https://au.oliuxue.com/studentnews/?page={page_number}"
        print(f'Accessing {url}')
        resp = requests.get(url, headers={"User-Agent": "MyBot/1.0"})
        time.sleep(2)
        soup = BeautifulSoup(resp.text, "lxml")

        # Extract all href links
        for a in soup.find_all("a", href=True):
            full_url = urljoin(url, a["href"])  # make relative URLs absolute
            # 只保留详情页：包含 .html，排除分页
            if (
                full_url.startswith("https://au.oliuxue.com/studentnews/") and
                full_url.endswith(".html") and
                "?page=" not in full_url
            ):
                links.add(full_url)

    return links


In [3]:
def load_web(url):
    print(f"Fetching {url}")
    downloaded = trafilatura.fetch_url(url)
    if not downloaded:
        print(f"Filtered out {url}: failed to fetch")
        return None

    data_json = trafilatura.extract(
        downloaded,
        output_format="json",
        with_metadata=True,
        include_comments=False,
        include_images=False
    )

    if not data_json:
        print(f"Filtered out {url}: extraction returned None")
        return None

    data = json.loads(data_json)

    # 严格过滤 title
    title = (data.get("title") or "").strip()
    if not title or title == "undefined":
        print(f"Filtered out {url}: invalid title")
        return None

    return data



In [4]:
def check_language(text):
    # detect language
    try:
        language = detect(text)
    except:
        return None

    # check chinese script type
    if language.startswith('zh'):
        has_simp = hanzidentifier.is_simplified(text)
        has_trad = hanzidentifier.is_traditional(text)
        
        if has_simp and not has_trad:
            return "simplified-chinese"
        elif has_trad and not has_simp:
            return "traditional-chinese"
        elif has_simp and has_trad:
            return 'mixed-chinese'

    # English language
    elif language.startswith("en"):
        return 'english'
    else:
        return language

In [5]:
def extract_source(url):
    parsed = urlparse(url)
    domain = parsed.netloc
    return domain

In [6]:
def process_data(raw_data, url):
    if not raw_data:
        return None

    title = (raw_data.get("title") or "").strip()
    if not title:
        print(f"Filtered out {url}: no title")
        return None

    text = (raw_data.get("text") or "").strip()
    if len(text) < 50:
        print(f"Filtered out {url}: text too short")
        return None
    if any(k in text for k in ["首页", "更多>>", "当前位置"]):
        print(f"Filtered out {url}: contains navigation keywords")
        return None

    tags = raw_data.get("tags")
    if tags is None:
        tags = []
    elif isinstance(tags, str):
        tags = [tags]

    return {
        "questions": [title],
        "text": text,
        "source": extract_source(url),
        "author": raw_data.get("author"),
        "post_date": raw_data.get("date"),
        "language": check_language(text),
        "created_at": raw_data.get("filedate"),
        "tags": tags,
        "link": url,
    }





In [7]:
def main():
    json_list = []
    urls = extract_TUN_XIAO_EDU_AU_link()
    print(f'URL collection succeeded, collected {len(urls)} URLs')

    seen = set()  # 去重
    for url in urls:
        if url in seen:
            continue
        seen.add(url)

        try:
            data_json = load_web(url)
            time.sleep(2)  # 避免请求过快
            if not data_json:
                print(f'No result fetched for {url}\n')
                continue

            result_json = process_data(data_json, url)
            if result_json:
                json_list.append(result_json)
                print(f'Successfully fetched: {url}\n')

        except Exception as e:
            print(f'Failed to fetch {url}: {e}\n')
            continue

    print('Fetch finished. Writing to JSON file...')
    with open("../data/YUN_XIAO_EDU_AU.json", "w", encoding="utf-8") as f:
        json.dump(json_list, f, ensure_ascii=False, indent=2)
    print('Done!')


In [8]:
if __name__ == '__main__':
    main()

Accessing https://au.oliuxue.com/studentnews/
Accessing https://au.oliuxue.com/studentnews/?page=1
Accessing https://au.oliuxue.com/studentnews/?page=2
Accessing https://au.oliuxue.com/studentnews/?page=3
Accessing https://au.oliuxue.com/studentnews/?page=4
Accessing https://au.oliuxue.com/studentnews/?page=5
Accessing https://au.oliuxue.com/studentnews/?page=6
Accessing https://au.oliuxue.com/studentnews/?page=7
Accessing https://au.oliuxue.com/studentnews/?page=8
Accessing https://au.oliuxue.com/studentnews/?page=9
Accessing https://au.oliuxue.com/studentnews/?page=10
Accessing https://au.oliuxue.com/studentnews/?page=11
Accessing https://au.oliuxue.com/studentnews/?page=12
Accessing https://au.oliuxue.com/studentnews/?page=13
URL collection succeeded, collected 218 URLs
Fetching https://au.oliuxue.com/studentnews/1553.html
Successfully fetched: https://au.oliuxue.com/studentnews/1553.html

Fetching https://au.oliuxue.com/studentnews/2278.html
Successfully fetched: https://au.oliuxue